# BTC NNUE - streamed trainingTrains the BetterThanCris NNUE straight from Stockfish `.binpack` files, withno intermediate corpus on disk. This is the same trainer and the same featureencoding used locally; only the data source differs, so a net produced here isdirectly comparable to one produced on the laptop.**Settings before running:** Notebook options -> Accelerator -> **GPU T4 x2**,and Internet **on**.**Do not pick P100.** It is Pascal (sm_60) and Kaggle's PyTorch build shipskernels only for sm_70 and later, so every CUDA op raises`no kernel image is available for execution on the device`. A P100 run failedthat way after 17 minutes, all of it spent downloading before the first CUDAcall. T4 is sm_75 and works. P100 has more than twice T4's memory bandwidth andthis step is bandwidth-bound, so it would have been the better card - but a cardtorch cannot launch a kernel on is worth nothing.**Runtime:** roughly 4-5 h for one pass over ~5B positions at L1=512.**Why L1=512 with 32 king buckets rather than a wider net.** Measured on theengine: 32 buckets cost 1.01x per node against 4 buckets, while L1=1024 costs1.38x against L1=512. Buckets change neither the number of active features(~32) nor the accumulator width, so only the weight table grows - they arecapacity we get for free, and width is capacity we pay for on every node. At32 buckets this is the same feature set Stockfish uses (HalfKAv2_hm: 32 kingbuckets x 11 piece planes x 64 squares = 22,528; ours is 32 x 12 x 64 =24,576).Buckets split the data 32 ways, so their risk is starvation - which is exactlywhy this belongs on Kaggle: ~5B positions here is 156M per bucket against 54Mon the laptop's corpus.**Eight output buckets, keyed on piece count**, as Stockfish does. One outputlayer has to map the accumulator to a score across every phase of the game, buta pawn up in a rook ending is not worth what a pawn up in a middlegame is.Separate output vectors let the net say so and cost nothing at inference - thedot product is the same length either way, only which weight vector is readchanges. Verified bit-identical between the engine and the independent numpyreference, and `test_accumulator` passes at this shape. Kaggle killsa session at 12 h, so keep `EPOCHS` at 3 or fewer. `net.npz` is written to `/kaggle/working` after every epochthat improves validation loss, and **`net_latest.npz` is rewritten every 5% ofan epoch** - so a session killed at the 12 h limit still leaves a usable netrather than nothing.

## 1. Check what GPU we actually got

In [ ]:
import subprocess, torchprint(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",                      "--format=csv"], capture_output=True,                     text=True).stdout)print("torch", torch.__version__, "cuda_available", torch.cuda.is_available())# torch.cuda.is_available() returns True on an architecture this build has no# kernels for, and the failure only surfaces at the first real CUDA op - which# is after the downloads. So launch an actual kernel here and stop now if it# cannot run, rather than 17 minutes and 33 GB later.assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 x2"name = torch.cuda.get_device_name(0)major, minor = torch.cuda.get_device_capability(0)supported = torch.cuda.get_arch_list()print(f"device {name}, compute capability sm_{major}{minor}")print(f"this torch was built for: {supported}")try:    probe = (torch.ones(1024, device="cuda") * 2).sum().item()    assert probe == 2048.0, probe    print("GPU KERNEL LAUNCH OK - safe to continue")except Exception as exc:    raise SystemExit(        f"GPU UNUSABLE: {name} is sm_{major}{minor}, this torch supports "        f"{supported}. Switch Accelerator to GPU T4 x2 and re-run. ({exc})")print(subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/temp"],                     capture_output=True, text=True).stdout)

## 2. Source files, written verbatim from the local repo

In [ ]:
%%writefile nnue_stream.py"""Training batches read straight out of .binpack files, with no corpus on disk.    from nnue_stream import BinpackStream    stream = BinpackStream(exe, ["a.binpack", "b.binpack"])    for feats, stm, score, result in stream.batches(16384):        ...The materialised path (binpack_convert -> binpack_prep -> nnue_train) turns 2Bpositions into 128 GB of .raw plus a 128 GB shuffle pass. That is affordable ona machine with a spare terabyte and several hours, and impossible on a hostednotebook with ~70 GB of disk in total. Here the only thing on disk is thebinpack, which holds those same 2B positions in 10 GB, so the corpus size stopsbeing a limit on how much data we can train on.**How the shuffle works, and why it is enough.** binpack rows arrive in gameorder, and consecutive plies of one game have near-identical evaluations, sotraining on the raw order would give a 16384-position batch an effective samplesize of a few dozen. The reader therefore fills a large buffer, permutes it, andserves batches from the permutation. A buffer of 4M positions spans roughly25,000 games, so a batch drawn from it is effectively a random sample; this isthe same approach bullet uses, and the reason the two-pass external shuffle isnot needed. What it does *not* give is a globally uniform permutation - aposition from the end of the last file can never land in the first batch - whichis why validation must come from separate files rather than from a tail slice.The reader runs on its own thread and stays a couple of buffers ahead, so thedecode and the host copy overlap GPU work instead of serialising with it."""import osimport queueimport subprocessimport threadingimport numpy as np# Field for field the layout of `struct Record` in tools/binpack_stream.cpp.# Built from a plain list so numpy packs it, giving the 69 bytes the C++ side# asserts on; an aligned dtype would silently insert padding and desynchronise# the whole stream after the first record.RECORD = np.dtype([("feats", "<u2", 32), ("count", "u1"), ("stm", "u1"),                   ("score", "<i2"), ("result", "i1")])assert RECORD.itemsize == 69, RECORD.itemsizeBUFFER_ROWS = int(os.environ.get("BTC_STREAM_BUFFER", "4000000"))QUEUE_DEPTH = 2class BinpackStream:    """Shuffled batches from a set of binpack files. One pass = one epoch."""    def __init__(self, exe, files, buffer_rows=BUFFER_ROWS, seed=None,                 max_rows=0):        # Absolute: CreateProcess rejects a relative path containing forward        # slashes, so "tools/binpack_stream.exe" launches from a shell and not        # from subprocess.        self.exe = os.path.abspath(exe)        self.files = [os.path.abspath(f) for f in files]        self.buffer_rows = buffer_rows        self.rng = np.random.default_rng(seed)        # A cap makes an "epoch" a fixed number of positions rather than a        # whole pass. With a corpus far larger than the time available that is        # the only way the cosine schedule can know how long it has to run.        self.max_rows = max_rows        self.rows_seen = 0        for path in self.files + [self.exe]:            if not os.path.exists(path):                raise FileNotFoundError(path)    def _reader(self, proc, out, stop):        """Decode fixed-size blocks off the pipe until it ends."""        want = self.buffer_rows * RECORD.itemsize        try:            while True:                # readinto on a bytearray, because read() on a pipe returns                # short reads whenever the writer flushes mid-block and a short                # block would misalign every record after it.                buf = bytearray(want)                got = 0                while got < want and not stop.is_set():                    chunk = proc.stdout.read(want - got)                    if not chunk:                        break                    buf[got:got + len(chunk)] = chunk                    got += len(chunk)                if got == 0 or stop.is_set():                    break                rows = got // RECORD.itemsize                out.put(np.frombuffer(bytes(buf[:rows * RECORD.itemsize]),                                      dtype=RECORD))                if got < want:                    break        except (ValueError, OSError):            # The consumer abandoned the generator and closed the pipe under            # us. That is a normal early exit, not a failure.            pass        finally:            out.put(None)    def batches(self, batch_size, drop_last=True):        """Yield (feats, stm, score, result) numpy arrays, shuffled."""        proc = subprocess.Popen([self.exe] + self.files,                                stdout=subprocess.PIPE,                                stderr=subprocess.DEVNULL,                                bufsize=0)        out = queue.Queue(maxsize=QUEUE_DEPTH)        stop = threading.Event()        thread = threading.Thread(target=self._reader, args=(proc, out, stop),                                  daemon=True)        thread.start()        try:            while True:                block = out.get()                if block is None:                    break                order = self.rng.permutation(len(block))                for start in range(0, len(block), batch_size):                    idx = order[start:start + batch_size]                    if len(idx) < batch_size and drop_last:                        continue                    rows = block[idx]                    self.rows_seen += len(idx)                    yield (rows["feats"], rows["stm"], rows["score"],                           rows["result"])                    if self.max_rows and self.rows_seen >= self.max_rows:                        return        finally:            # Order matters. Killing the writer first makes the reader's            # pending read return empty rather than block, and draining the            # queue unwedges it if it is parked on a full put; only then is it            # safe to close the pipe.            stop.set()            proc.terminate()            while not out.empty():                out.get_nowait()            thread.join(timeout=30)            proc.stdout.close()            proc.wait(timeout=30)

In [ ]:
%%writefile nnue_train.py"""Train and quantise a (768 -> L1)x2 -> 1 NNUE. Run with the CUDA venv:    .venv_nnue\\Scripts\\python.exe nnue_train.py <data_dir> <out_dir> [L1] [epochs]Follows the configuration the bullet trainer and nnue-pytorch converge on; everyconstant here has a primary source behind it and the reasoning is indocs/TEST_PLAN.md. The parts that are easy to get quietly wrong, and thereforecarry the most comment, are the perspective ordering, the sign convention, andthe quantisation.Feature indices are stored by nnue_data.py as    index = 384 * colour + 64 * piece_type + square,   square 0 = a8which is exactly the white-perspective formula. The black perspective flips theboard vertically and swaps the colour plane:    black_index = 384 * (1 - colour) + 64 * piece_type + (square ^ 56)`square ^ 56` is a vertical flip in either square convention because it invertsthe rank bits, so this transfers unchanged from sources that use square 0 = a1.What matters is only that training and inference agree, and inference reads thesame table."""import osimport sysimport timeimport numpy as npimport torchimport torch.nn as nnPAD = 65535# King buckets. Without them every position shares one 768-feature table, so# the net has to average over all king placements; bucketing lets it learn# king-relative structure, which is most of what a hand-crafted evaluation# spends its king-safety terms on.## Buckets are the cheapest capacity we have. They change neither the number of# active features (~32) nor the accumulator width, so only the weight table# grows: measured on the engine, a node costs 1414 ns at 4 buckets and 1430 ns# at 32. Width is the expensive kind - L1=1024 costs 1.38x per node against# L1=512. **Prefer buckets over width.**## At 32 this is the feature set Stockfish uses. HalfKAv2_hm is 32 king buckets# x 11 piece planes x 64 squares = 22,528 features per perspective; ours is# 32 x 12 x 64 = 24,576, the extra plane being the own king, which a 32-bucket# index already determines and which therefore acts as a per-bucket bias.## The real limit is data, not speed: buckets split the corpus N ways, so each# bucket's weights see 1/N of the positions. At 32 buckets, 1.74B positions is# 54M per bucket against the shipped net's 70M - hence the sweep over 4/16/32# rather than jumping straight to the largest.## The 50 MB unzipped cap binds at the top: 32 buckets is 25.2 MB at L1=512 and# 37.7 MB at L1=768, but 50.3 MB at L1=1024, which does not fit.NUM_BUCKETS = int(os.environ.get("BTC_KING_BUCKETS", "4"))def _bucket_table(count):    """King square (0 = a8, perspective orientation) -> bucket.    With horizontal mirroring the king is always canonicalised onto files a-d,    so the bucket is simply its canonical file and four buckets cover the board.    Without mirroring, files e-h would need four more buckets to say the same    thing, splitting the data across mirror-image positions that are strategically    identical - which is exactly the waste mirroring exists to remove.    Entry `square` is read *after* canonicalisation, so only files 0-3 are ever    looked up; the rest are filled consistently so a stale index cannot silently    read a wrong bucket.    Beyond four, rank resolution is added on top of the file, so the counts form    a hierarchy: 4 splits on file alone, 8 adds the board half, 16 the quarter,    32 the exact canonical square - which is what the HalfKP-style feature sets    every strong engine uses amount to. **A bucket refines the previous level    rather than replacing it**, so the 4-bucket networks already measured stay    directly comparable to the wider ones.    This is the cheapest capacity available to us: buckets change neither the    number of active features (~32) nor the accumulator width, so the per-node    cost is flat - measured 1414 ns at 4 buckets against 1430 ns at 32, while    the table grows 3.1 MB to 25.2 MB. Width, by contrast, is paid on every    node. The limit is the 50 MB unzipped cap: 32 buckets fits at L1=512    (25.2 MB) and does not at L1=1024."""    table = np.zeros(64, dtype=np.int32)    if count == 1:        return table    for square in range(64):        file_index = square % 8        if file_index >= 4:            file_index = 7 - file_index        rank = square // 8        if count <= 4:            table[square] = file_index        else:            # count // 4 rank groups spread over 8 ranks:            # 8 -> rank half, 16 -> rank quarter, 32 -> exact rank            divisor = 32 // count            table[square] = (rank // divisor) * 4 + file_index    return table# WDL blend. The training target is##     lambda * sigmoid(cp / SCALE) + (1 - lambda) * (result + 1) / 2## lambda = 1.0 is pure evaluation, which is all the Lichess evaluations database# can support because it has no game results. Binpack data carries the outcome# of the game the position came from, and blending it in is the single# largest known gain in NNUE training practice - tcheran measured +53.33 +-15.20# and then a further +27.05 +-9.85 from raising the WDL proportion.## The reason it works: an evaluation says how good a position looks to a search,# while the result says how often it is actually converted. They differ most in# exactly the positions that decide games - drawish endings a search scores as# +1.5, sharp positions a search scores as equal. Defaults to pure eval so that# eval-only datasets behave as before.WDL_LAMBDA = float(os.environ.get("BTC_WDL_LAMBDA", "1.0"))# Output buckets, selected by piece count. One output layer has to express a# single mapping from accumulator to score across every phase of the game, but# a pawn-up rook ending and a pawn-up middlegame are not worth the same number# of centipawns. Eight separate output vectors let the net say so.## **This is free at inference.** The layer is a dot product of 2*L1 terms# either way; a bucket only changes which weight vector is read. Stockfish uses# eight, keyed the same way.OUT_BUCKETS = int(os.environ.get("BTC_OUT_BUCKETS", "1"))def out_bucket_of(piece_count, out_buckets):    """Piece count (2..32) -> output bucket. Must match btc_nnue._out_bucket.    Takes a tensor or an int, so the trainer and the tests share one definition    of the mapping rather than two that can drift apart."""    if out_buckets <= 1:        return piece_count * 0    idx = (piece_count - 1) * out_buckets // 32    if hasattr(idx, "clamp"):        return idx.clamp(0, out_buckets - 1)    return max(0, min(out_buckets - 1, idx))QA = 255            # feature transformer / accumulator scaleQB = 64             # output layer scaleSCALE = 400         # network float output -> centipawns, and the loss sigmoidCLIP = 1.98         # weight clip; keeps the int16 accumulator from overflowingBATCH = 16384LR = 1e-3FINAL_LR = 1e-3 * (0.3 ** 5)WEIGHT_DECAY = 0.01VAL_FRACTION = 0.01class Nnue(nn.Module):    """(768 -> L1) x 2 -> 1 with SCReLU.    The feature transformer is an EmbeddingBag in sum mode, which *is* the    accumulator: its weight matrix is (768, L1) row-major, exactly the layout    the numba inference reads, so one row is one contiguous run of L1 weights."""    def __init__(self, l1, buckets=1, out_buckets=1):        super().__init__()        self.l1 = l1        self.buckets = buckets        self.out_buckets = out_buckets        rows = buckets * 768        # One extra row: a permanently-zero padding row, so a short piece list        # sums to the same accumulator as a full one. padding_idx pins it at        # zero and excludes it from gradients, so weight decay cannot drift it        # away from zero over 87k steps.        self.ft = nn.EmbeddingBag(rows + 1, l1, mode="sum", padding_idx=rows)        self.ft_bias = nn.Parameter(torch.zeros(l1))        self.out = nn.Linear(2 * l1, out_buckets)        nn.init.normal_(self.ft.weight, std=0.01)    def forward(self, white_idx, black_idx, stm, out_bucket=None):        acc_w = self.ft(white_idx) + self.ft_bias        acc_b = self.ft(black_idx) + self.ft_bias        # Side to move FIRST. This is what lets the net learn tempo, and it        # makes the output side-to-move relative, which is what negamax wants.        # stm is 1.0 when white is to move.        stm = stm.unsqueeze(1)        hidden = stm * torch.cat([acc_w, acc_b], dim=1) \            + (1.0 - stm) * torch.cat([acc_b, acc_w], dim=1)        hidden = torch.clamp(hidden, 0.0, 1.0) ** 2      # SCReLU        scores = self.out(hidden)        if self.out_buckets == 1:            return scores.squeeze(1)        # Every bucket's score is computed and one is selected. Wasteful in        # training and irrelevant there (the layer is tiny next to the        # embedding); inference reads only the selected vector.        return scores.gather(1, out_bucket.unsqueeze(1)).squeeze(1)def _king_square(feats, pad, base):    """Square of the king whose feature plane starts at `base`.    The data has exactly one king per side per row - test_nnue and the    extraction verifier both assert it - so masking to that plane and summing    recovers the single index without a search."""    mask = (feats >= base) & (feats < base + 64) & ~pad    return (feats * mask).sum(dim=1) - basedef build_perspectives(feats, device, table, buckets):    """(white_idx, black_idx) index tensors from the stored white-perspective    indices, offset by each perspective's own king bucket.    Each perspective buckets on *its own* king, the black one after the    `square ^ 56` flip. That is what keeps the colour-mirror symmetry exact:    mirroring swaps which king each perspective sees, so the two accumulators    swap and the output is unchanged."""    feats = feats.to(device=device, dtype=torch.long)    pad = feats == PAD    colour = torch.div(feats, 384, rounding_mode="floor")    piece = torch.div(feats % 384, 64, rounding_mode="floor")    square = feats % 64    if buckets == 1:        white = feats.clone()        black = 384 * (1 - colour) + 64 * piece + (square ^ 56)        white[pad] = 768        black[pad] = 768        return white, black    white_king = _king_square(feats, pad, 5 * 64)    black_king = _king_square(feats, pad, 11 * 64) ^ 56    # Mirror each perspective independently, onto files a-d, keyed on that    # perspective's own king. `^ 7` inverts the file bits.    white_flip = ((white_king % 8) >= 4).long().unsqueeze(1) * 7    black_flip = ((black_king % 8) >= 4).long().unsqueeze(1) * 7    white_square = square ^ white_flip    black_square = (square ^ 56) ^ black_flip    white = 384 * colour + 64 * piece + white_square    black = 384 * (1 - colour) + 64 * piece + black_square    white = white + table[white_king].unsqueeze(1) * 768    black = black + table[black_king].unsqueeze(1) * 768    rows = buckets * 768    white[pad] = rows    black[pad] = rows    return white, blackdef _step(model, opt, sched, arrays, device, table, train):    """One batch, from four numpy arrays to a scalar loss.    Both the memory-mapped path and the streaming path funnel through here, so    the target construction, the sign conventions and the weight clamp cannot    drift between a net trained one way and a net trained the other. That    matters because the two are compared against each other."""    raw_feats, raw_stm, raw_score, raw_result = arrays    batch_feats = torch.from_numpy(np.asarray(raw_feats).astype(np.int64))    batch_stm = torch.from_numpy(        np.asarray(raw_stm).astype(np.int64)).to(device).float()    batch_score = torch.from_numpy(        np.asarray(raw_score).astype(np.float32)).to(device)    batch_result = None    if raw_result is not None:        batch_result = torch.from_numpy(            np.asarray(raw_result).astype(np.float32)).to(device)    white, black = build_perspectives(batch_feats, device, table, model.buckets)    # stored stm is 0 for white to move; the model wants 1.0 for white    white_to_move = 1.0 - batch_stm    target = torch.sigmoid(batch_score / SCALE)    if batch_result is not None and WDL_LAMBDA < 1.0:        # result is -1/0/+1 side-to-move relative, the same convention as the        # score, so it maps to a win probability with (r + 1) / 2.        outcome = (batch_result + 1.0) * 0.5        target = WDL_LAMBDA * target + (1.0 - WDL_LAMBDA) * outcome    bucket = None    if model.out_buckets > 1:        # Piece count is the number of non-padding features, so it needs no        # extra column in the data.        counts = (batch_feats != PAD).sum(dim=1).to(device)        bucket = out_bucket_of(counts, model.out_buckets)    with torch.set_grad_enabled(train):        pred = torch.sigmoid(model(white, black, white_to_move, bucket))        loss = ((pred - target) ** 2).mean()    if train:        opt.zero_grad(set_to_none=True)        loss.backward()        opt.step()        sched.step()        # Hard clamp after every step, not a penalty in the loss. This is the        # entire reason the int16 accumulator cannot overflow:        # 505 (bias) + 32 * 505 = 16,665 against a 32,767 ceiling.        with torch.no_grad():            for p in (model.ft.weight, model.ft_bias,                      model.out.weight, model.out.bias):                p.clamp_(-CLIP, CLIP)    with torch.no_grad():        mae = float((pred - target).abs().sum())        scalar = float(loss.detach())    return scalar, maeclass _Progress:    """Percent, throughput and running loss during an epoch.    A full pass over 2B positions is hours long; without this a run is silent    until it ends, so a stall, a thermal throttle or a wrong row count is only    discovered after the time has already been spent."""    def __init__(self, total, every):        self.total = total        self.every = max(1, every)        self.started = time.time()    def update(self, step, count, total_loss):        """Print a progress line. Returns True when it did, so the caller can        hang periodic work off the same cadence."""        if step % self.every:            return False        rate = count / max(time.time() - self.started, 1e-9)        pct = f"{100 * count / self.total:5.1f}%" if self.total else "  ---"        print(f"    {pct}  {count / 1e6:8.1f}M seen  {rate / 1e6:5.3f} Mpos/s  "              f"loss {total_loss / max(count, 1):.6f}", flush=True)        return Truedef run_epoch(model, opt, sched, data, device, train, table):    """One pass over memory-mapped arrays.    **Contiguous batches, shuffled batch order** - not random indices.    Indexing a memory-mapped file with 16384 scattered row numbers is fine while    the file fits in page cache and collapses into random I/O when it does not.    feats is 64 bytes per position, so 300M positions is 19 GB and random access    would make training I/O-bound rather than GPU-bound. Reading each batch as    one contiguous slice keeps the access sequential at any file size, which is    what makes scaling the dataset possible at all.    Randomness is not lost: binpack_prep.py shuffles the rows on disk, so a    contiguous slice is already a random sample of the corpus, and shuffling the    *order* of batches each epoch stops the model seeing them in a fixed    sequence. This is what bullet does for the same reason."""    feats, stm, score, result = data    total = 0.0    count = 0    mae = 0.0    model.train(train)    starts = list(range(0, len(score), BATCH))    if train:        np.random.default_rng().shuffle(starts)    progress = _Progress(len(score), len(starts) // 20)    for step, start in enumerate(starts):        stop = min(start + BATCH, len(score))        size = stop - start        # Partial final batch included. Dropping it silently made the        # validation loop skip entirely whenever the held-out set was smaller        # than a batch, and report a loss of exactly 0.        if size < 2:            continue        arrays = (feats[start:stop], stm[start:stop], score[start:stop],                  None if result is None else result[start:stop])        loss, batch_mae = _step(model, opt, sched, arrays, device, table, train)        total += loss * size        count += size        mae += batch_mae        if train:            progress.update(step, count, total)    return total / max(count, 1), mae / max(count, 1)def run_epoch_stream(model, opt, sched, stream, device, train, table,                     expected=0, checkpoint=None):    """One pass over binpack files, with nothing materialised on disk.    `expected` is only used to render a percentage and to space the progress    reports; the stream itself knows how many positions it will produce only    once it has produced them.    `checkpoint` is called at each progress report. A streamed epoch over    billions of positions runs for hours, and a hosted notebook is killed at a    fixed wall-clock limit - so an epoch-end-only save means a run that is    stopped at 99% produces *nothing*. This writes the current weights out    every time it reports, which costs one 6 MB file write per twentieth of an    epoch and turns a killed session into a usable net."""    total = 0.0    count = 0    mae = 0.0    model.train(train)    progress = _Progress(expected, max(1, expected // BATCH // 20) if expected                         else 200)    for step, arrays in enumerate(stream.batches(BATCH)):        loss, batch_mae = _step(model, opt, sched, arrays, device, table, train)        size = len(arrays[2])        total += loss * size        count += size        mae += batch_mae        if train and progress.update(step, count, total) and checkpoint:            checkpoint()    return total / max(count, 1), mae / max(count, 1), countdef quantise(model, out_dir, l1, table, name="net.npz"):    """Export int16 weights, asserting nothing overflows rather than wrapping."""    rows = model.buckets * 768    ft_w = model.ft.weight.detach().cpu().numpy()[:rows]    ft_b = model.ft_bias.detach().cpu().numpy()    # (out_buckets, 2 * l1) and (out_buckets,), even when there is one bucket,    # so the engine reads one shape rather than two.    out_w = model.out.weight.detach().cpu().numpy()    out_b = model.out.bias.detach().cpu().numpy()    ft_w_q = np.round(ft_w * QA)    ft_b_q = np.round(ft_b * QA)    out_w_q = np.round(out_w * QB)    out_b_q = np.round(out_b * QA * QB)    assert out_w_q.shape == (model.out_buckets, 2 * l1), out_w_q.shape    limit = np.iinfo(np.int16).max    assert np.abs(ft_w_q).max() <= limit, "feature weights overflow int16"    assert np.abs(ft_b_q).max() <= limit, "feature bias overflows int16"    assert np.abs(out_w_q).max() <= limit, "output weights overflow int16"    worst = abs(ft_b_q).max() + 32 * abs(ft_w_q).max()    assert worst <= limit, f"accumulator can reach {worst}, over int16"    np.savez(os.path.join(out_dir, name),             ft_w=ft_w_q.astype(np.int16), ft_b=ft_b_q.astype(np.int16),             out_w=out_w_q.astype(np.int16),             out_b=out_b_q.astype(np.int32), l1=np.int32(l1),             out_buckets=np.int32(model.out_buckets),             qa=np.int32(QA), qb=np.int32(QB), scale=np.int32(SCALE),             buckets=np.int32(model.buckets),             bucket_table=table.cpu().numpy().astype(np.int32))    if name == "net.npz":        print(f"quantised: worst-case accumulator {int(worst)} of {limit}")def _build(l1, device):    """Model, optimiser and the zeroed padding row, shared by both modes."""    model = Nnue(l1, NUM_BUCKETS, OUT_BUCKETS).to(device)    # pad row 768 must stay zero and out of the optimiser's way    with torch.no_grad():        model.ft.weight[NUM_BUCKETS * 768:].zero_()    opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.999),                            eps=1e-8, weight_decay=WEIGHT_DECAY)    return model, optdef _record(out_dir, l1, table, model, epoch, epochs, tr, va, va_mae, best):    """Checkpoint on improvement and print the epoch line."""    flag = ""    if va < best:        best = va        torch.save(model.state_dict(), os.path.join(out_dir, "best.pt"))        quantise(model, out_dir, l1, table)        flag = "  *"    print(f"epoch {epoch + 1:3d}/{epochs}  train {tr:.6f}  "          f"val {va:.6f}  val_sig_mae {va_mae:.4f}{flag}", flush=True)    return bestdef _binpacks(path):    """The binpack files named by `path`, which may be one file or a directory."""    if os.path.isfile(path):        return [path]    if not os.path.isdir(path):        return []    found = sorted(os.path.join(path, f) for f in os.listdir(path)                   if f.endswith(".binpack"))    return founddef train_streamed(files, out_dir, l1, epochs):    """Train directly from binpacks. Nothing but the binpacks touches disk.    Validation needs its own files rather than a tail slice: the stream shuffles    within a buffer, not globally, so a tail slice would be a systematically    later - and therefore different - sample of the corpus."""    from nnue_stream import BinpackStream    exe = os.environ.get("BTC_STREAM_EXE", "tools/binpack_stream")    val_files = _binpacks(os.environ.get("BTC_VAL_BINPACK", ""))    if not val_files:        if len(files) < 2:            raise SystemExit(                "streaming needs a validation binpack: set BTC_VAL_BINPACK, or "                "pass a directory holding more than one .binpack")        val_files, files = files[-1:], files[:-1]    # The validation file usually lives in the same directory as the training    # files, so passing that directory would otherwise train on it and make    # every validation number meaningless without failing.    held = {os.path.abspath(f) for f in val_files}    files = [f for f in files if os.path.abspath(f) not in held]    if not files:        raise SystemExit("no training files left after holding out validation")    # Only used for the progress percentage and the cosine schedule length; the    # stream cannot know its own length until it has finished.    expected = int(os.environ.get("BTC_STREAM_ROWS", "0"))    val_rows = int(os.environ.get("BTC_STREAM_VAL_ROWS", "4000000"))    wdl = "eval only" if WDL_LAMBDA >= 1.0 else f"WDL lambda {WDL_LAMBDA}"    print(f"streaming {len(files)} binpack(s), validating on "          f"{len(val_files)}, L1={l1}, {NUM_BUCKETS} king buckets, "          f"{OUT_BUCKETS} output buckets, {epochs} epochs, {wdl}")    for path in files + val_files:        print(f"  {os.path.getsize(path) / 1e9:7.2f} GB  {path}")    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    table = torch.from_numpy(_bucket_table(NUM_BUCKETS)).to(device)    model, opt = _build(l1, device)    steps = max(1, (expected or 500_000_000) // BATCH) * epochs    sched = torch.optim.lr_scheduler.CosineAnnealingLR(        opt, T_max=steps, eta_min=FINAL_LR)    best = float("inf")    for epoch in range(epochs):        train_stream = BinpackStream(exe, files, seed=epoch,                                     max_rows=expected)        def save_latest():            """Mid-epoch insurance against a hosted session being killed."""            quantise(model, out_dir, l1, table, "net_latest.npz")        tr, _, seen = run_epoch_stream(model, opt, sched, train_stream, device,                                       True, table, expected, save_latest)        val_stream = BinpackStream(exe, val_files, seed=0,                                   max_rows=val_rows)        va, va_mae, _ = run_epoch_stream(model, opt, sched, val_stream, device,                                         False, table)        print(f"  epoch {epoch + 1} streamed {seen:,} training positions")        best = _record(out_dir, l1, table, model, epoch, epochs, tr, va,                       va_mae, best)def train_mapped(data_dir, out_dir, l1, epochs):    """Train from the .npy arrays binpack_prep.py writes."""    feats = np.load(os.path.join(data_dir, "feats.npy"), mmap_mode="r")    stm = np.load(os.path.join(data_dir, "stm.npy"), mmap_mode="r")    score = np.load(os.path.join(data_dir, "score.npy"), mmap_mode="r")    result_path = os.path.join(data_dir, "result.npy")    result = np.load(result_path, mmap_mode="r")         if os.path.exists(result_path) else None    n = len(score)    # Ablations (bucket count, WDL lambda, L1) only need enough data to rank    # configurations, not the whole corpus. Capping rows makes each comparison    # run in a fraction of the time; the winning configuration is then retrained    # on everything. The data is shuffled on disk, so a prefix is a random    # sample rather than a biased one.    cap = int(os.environ.get("BTC_MAX_ROWS", "0"))    if cap and cap < n:        n = cap        feats, stm, score = feats[:n], stm[:n], score[:n]        if result is not None:            result = result[:n]    # The data is already shuffled on disk, so a tail slice is a random sample;    # taking it explicitly anyway so this does not depend on that holding.    split = n - int(n * VAL_FRACTION)    wdl = "eval only" if result is None or WDL_LAMBDA >= 1.0         else f"WDL lambda {WDL_LAMBDA}"    print(f"{n:,} positions, {n - split:,} held out, L1={l1}, "          f"{NUM_BUCKETS} king buckets, {OUT_BUCKETS} output buckets, "          f"{epochs} epochs, {wdl}")    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    table = torch.from_numpy(_bucket_table(NUM_BUCKETS)).to(device)    model, opt = _build(l1, device)    steps = max(1, (split // BATCH)) * epochs    sched = torch.optim.lr_scheduler.CosineAnnealingLR(        opt, T_max=steps, eta_min=FINAL_LR)    train_data = (feats[:split], stm[:split], score[:split],                  None if result is None else result[:split])    val_data = (feats[split:], stm[split:], score[split:],                None if result is None else result[split:])    best = float("inf")    for epoch in range(epochs):        tr, _ = run_epoch(model, opt, sched, train_data, device, True, table)        va, va_mae = run_epoch(model, opt, sched, val_data, device, False,                               table)        best = _record(out_dir, l1, table, model, epoch, epochs, tr, va,                       va_mae, best)def main():    data, out_dir = sys.argv[1], sys.argv[2]    l1 = int(sys.argv[3]) if len(sys.argv) > 3 else 256    epochs = int(sys.argv[4]) if len(sys.argv) > 4 else 30    os.makedirs(out_dir, exist_ok=True)    files = _binpacks(data)    if files:        train_streamed(files, out_dir, l1, epochs)    else:        train_mapped(data, out_dir, l1, epochs)if __name__ == "__main__":    main()

In [ ]:
%%writefile binpack_stream.cpp// Stream a Stockfish .binpack straight into training batches, with no// intermediate files at all.//// Build (Linux, from the repo root)://   g++ -O2 -std=c++20 -I<nnue-pytorch>/data_loader/cpp/lib \//       tools/binpack_stream.cpp -o tools/binpack_stream//// Run://   tools/binpack_stream a.binpack b.binpack ... > pipe//// **Why this exists alongside binpack_convert.** The batch converter// materialises the corpus: 2B positions become 128 GB of .raw, then a two-pass// external shuffle writes another 128 GB. That is fine on a machine with a// spare terabyte and five hours, and impossible anywhere else - a Kaggle// session has about 70 GB of disk in total. Streaming removes the corpus size// from the equation entirely: the only thing on disk is the binpack itself,// which holds 2B positions in 10 GB. It also removes the shuffle pass, because// the reader shuffles in RAM.//// The filters, conventions and record fields are deliberately identical to// binpack_convert.cpp so that a streamed net and a materialised net are// trained on the same distribution and remain comparable. The book filter,// which lives in binpack_prep.py for the materialised path, has to move here// because there is no later pass to apply it.//// This is local tooling and is never shipped; package.py has an explicit file// list. The rules ban native binaries in the submission, not on the machine// that prepares training data.#include <cstdio>#include <cstdint>#include <cstdlib>#include <cmath>#include <ctime>#include <string>#include <vector>#ifdef _WIN32#include <io.h>#include <fcntl.h>static inline struct tm* gmtime_r(const time_t* clock, struct tm* result) {    return gmtime_s(result, clock) == 0 ? result : nullptr;}#endif#include "nnue_training_data_stream.h"using namespace binpack;using namespace chess;static const int MAX_PIECES = 32;static const std::uint16_t PAD = 65535;static const int MAX_EVAL = 10000;static const int MIN_PIECES = 4;// One position, packed to 69 bytes with no padding, matching the numpy// structured dtype in nnue_stream.py field for field.#pragma pack(push, 1)struct Record {    std::uint16_t feats[MAX_PIECES];    std::uint8_t count;    std::uint8_t stm;    std::int16_t score;    std::int8_t result;};#pragma pack(pop)static_assert(sizeof(Record) == 69, "record must be packed to 69 bytes");struct Counters {    std::uint64_t seen = 0, kept = 0;    std::uint64_t eval = 0, pieces = 0, check = 0, tactical = 0, book = 0;};// Both kings and all four rooks still at home with full material: the// signature of a position still in the opening book. Leela self-play games all// start from the same place, so each game donates its opening and the corpus// ends up 1.35% literal start positions. Square 0 = a8, feature = 64 * piece +// square, so e1 = 60, a1 = 56, h1 = 63, e8 = 4, a8 = 0, h8 = 7.static bool is_book(const std::uint16_t* row, int count) {    if (count != 32) return false;    bool wk = false, wra = false, wrh = false;    bool bk = false, bra = false, brh = false;    for (int i = 0; i < count; ++i) {        std::uint16_t f = row[i];        if (f == 5 * 64 + 60) wk = true;        else if (f == 3 * 64 + 56) wra = true;        else if (f == 3 * 64 + 63) wrh = true;        else if (f == 11 * 64 + 4) bk = true;        else if (f == 9 * 64 + 0) bra = true;        else if (f == 9 * 64 + 7) brh = true;    }    return wk && wra && wrh && bk && bra && brh;}// Fills `out` from one entry, or returns false if the entry is filtered out.static bool encode(const TrainingDataEntry& e, Record& out, Counters& c) {    if (std::abs((int)e.score) > MAX_EVAL) { ++c.eval; return false; }    // A capture or promotion is about to invalidate the static evaluation,    // which is what makes such positions bad training targets.    if (e.isCapturingMove() || e.move.promotedPiece != Piece::none()) {        ++c.tactical;        return false;    }    if (e.isInCheck()) { ++c.check; return false; }    int count = 0;    for (int i = 0; i < MAX_PIECES; ++i) out.feats[i] = PAD;    for (Square sq : e.pos.piecesBB()) {        if (count >= MAX_PIECES) { ++c.pieces; return false; }        Piece p = e.pos.pieceAt(sq);        int piece = 6 * ordinal(p.color()) + ordinal(p.type());        out.feats[count++] = (std::uint16_t)(64 * piece + (ordinal(sq) ^ 56));    }    if (count < MIN_PIECES) { ++c.pieces; return false; }    if (is_book(out.feats, count)) { ++c.book; return false; }    out.count = (std::uint8_t)count;    out.stm = (std::uint8_t)ordinal(e.pos.sideToMove());    out.score = (std::int16_t)e.score;    out.result = (std::int8_t)e.result;    return true;}static void run_file(const std::string& path, Counters& c,                     std::vector<Record>& buf) {    auto stream = training_data::open_sfen_input_file(path, false);    if (!stream) {        std::fprintf(stderr, "cannot open %s\n", path.c_str());        std::exit(1);    }    while (true) {        auto entry = stream->next();        if (!entry.has_value()) break;        ++c.seen;        Record rec;        if (!encode(*entry, rec, c)) continue;        buf.push_back(rec);        ++c.kept;        if (buf.size() == buf.capacity()) {            std::fwrite(buf.data(), sizeof(Record), buf.size(), stdout);            buf.clear();        }        if (c.kept % 20000000 == 0) {            std::fprintf(stderr, "streamed %llu of %llu seen\n",                         (unsigned long long)c.kept,                         (unsigned long long)c.seen);            std::fflush(stderr);        }    }}int main(int argc, char** argv) {    if (argc < 2) {        std::fprintf(stderr, "usage: binpack_stream in.binpack [more...]\n");        return 1;    }#ifdef _WIN32    // stdout must not translate 0x0A into 0x0D 0x0A or every record shifts.    _setmode(_fileno(stdout), _O_BINARY);#endif    Counters c;    std::vector<Record> buf;    buf.reserve(16384);    for (int i = 1; i < argc; ++i) run_file(argv[i], c, buf);    if (!buf.empty())        std::fwrite(buf.data(), sizeof(Record), buf.size(), stdout);    std::fflush(stdout);    std::fprintf(stderr,                 "done: kept %llu of %llu seen\n"                 "dropped: eval %llu, pieces %llu, check %llu, "                 "tactical %llu, book %llu\n",                 (unsigned long long)c.kept, (unsigned long long)c.seen,                 (unsigned long long)c.eval, (unsigned long long)c.pieces,                 (unsigned long long)c.check, (unsigned long long)c.tactical,                 (unsigned long long)c.book);    return 0;}

In [ ]:
%%writefile verify_binpack.py"""Check a .binpack decodes to the conventions we train on, before training.    python verify_binpack.py <binpack> [more.binpack ...]Every silent disaster this project has had came from data that decoded intosomething *plausible* but wrong: the Lichess extraction had a sign conventioninverted and looked fine until the sign-agreement test was run. A net trained ona board it will never see costs the GPU hours and gives no error, so the cost ofskipping this check is the whole run.The decisive number is **score/result sign agreement**. If the evaluation andthe game outcome are both side-to-move relative, they agree far more often thanchance in positions with a clear advantage. If one were White-relative instead,agreement would sit near 50% and nothing else in the file would look wrong."""import osimport sysimport numpy as npfrom nnue_stream import BinpackStreamEXE = os.environ.get("BTC_STREAM_EXE", "tools/binpack_stream.exe")ROWS = int(os.environ.get("BTC_VERIFY_ROWS", "2000000"))BATCH = 16384def collect(path, rows):    stream = BinpackStream(EXE, [path], buffer_rows=min(rows, 500000), seed=0,                           max_rows=rows)    parts = [[], [], [], []]    for feats, stm, score, result in stream.batches(BATCH):        parts[0].append(feats.astype(np.int32))        parts[1].append(stm.astype(np.int32))        parts[2].append(score.astype(np.int32))        parts[3].append(result.astype(np.int32))    return [np.concatenate(p) for p in parts]def check(path, rows):    feats, stm, score, result = collect(path, rows)    valid = feats != 65535    counts = valid.sum(axis=1)    problems = []    if feats[valid].max() >= 768 or feats[valid].min() < 0:        problems.append("feature index outside 0..767")    kings_w = ((feats >= 5 * 64) & (feats < 6 * 64)).sum(axis=1)    kings_b = ((feats >= 11 * 64) & (feats < 12 * 64)).sum(axis=1)    if not ((kings_w == 1).all() and (kings_b == 1).all()):        problems.append("not exactly one king per side")    if counts.min() < 4 or counts.max() > 32:        problems.append("piece count outside 4..32")    if not set(np.unique(result).tolist()) <= {-1, 0, 1}:        problems.append("result outside {-1,0,+1}")    if set(np.unique(stm).tolist()) != {0, 1}:        problems.append("side to move is not {0,1}")    # The one that catches a flipped convention. Drawn games are excluded:    # sign(0) matches nothing and half of all games are drawn, so counting them    # caps agreement near 65% and makes correct data look inverted.    clear = (np.abs(score) > 200) & (result != 0)    agree = float(np.mean(np.sign(score[clear]) == np.sign(result[clear]))) \        if clear.any() else float("nan")    if not (agree > 0.75):        problems.append(f"score/result sign agreement {agree:.3f} - one of "                        f"them is probably not side-to-move relative")    # Duplicate rate, which is what the v6-dd suffix claims to have reduced.    packed = np.ascontiguousarray(feats).view(        np.dtype((np.void, feats.shape[1] * feats.dtype.itemsize)))    unique = len(np.unique(packed))    print(f"{os.path.basename(path)}")    print(f"  rows sampled        {len(counts):,}")    print(f"  pieces  mean        {counts.mean():.2f}  "          f"(min {counts.min()}, max {counts.max()})")    print(f"  score   mean        {score.mean():+.1f}  "          f"(min {score.min()}, max {score.max()})")    print(f"  result  mean        {result.mean():+.4f}  "          f"(win {np.mean(result > 0):.3f}, draw {np.mean(result == 0):.3f})")    print(f"  sign agreement      {agree:.3f}  on "          f"{int(clear.sum()):,} decisive rows with |score| > 200")    print(f"  unique positions    {100 * unique / len(counts):.2f}%")    print(f"  verdict             "          f"{'OK' if not problems else 'REJECT - ' + '; '.join(problems)}")    return not problemsdef main():    if len(sys.argv) < 2:        print(__doc__)        return 1    ok = True    for path in sys.argv[1:]:        ok &= check(path, ROWS)        print()    print("all files usable" if ok else "AT LEAST ONE FILE REJECTED")    return 0 if ok else 1if __name__ == "__main__":    sys.exit(main())

## 3. Build the reader`nnue-pytorch` is cloned only for its binpack decoder headers - we do not useits trainer, its model or its networks. Needs `-std=c++20`; the headers use`std::predicate`.

In [ ]:
!which zstd > /dev/null || apt-get -qq install -y zstd > /dev/null!git clone --depth 1 -q https://github.com/official-stockfish/nnue-pytorch.git /kaggle/temp/nnue-pytorch && echo CLONE OK!g++ -O2 -std=c++20 -I /kaggle/temp/nnue-pytorch/data_loader/cpp/lib     binpack_stream.cpp -o binpack_stream && echo BUILD OK!./binpack_stream 2>&1 | head -2

## 4. Fetch training dataOne month for training, a different month for validation. Validation must comefrom separate files: the stream shuffles within a buffer rather than globally,so a tail slice would be a systematically later - and therefore different -sample. Each archive is deleted right after it is decompressed, to stay insideKaggle's disk quota.

In [ ]:
import os, subprocessHOST = "https://huggingface.co/datasets/linrock/test80-2022/resolve/main"TRAIN_Z = ['test80-2022-06-jun-16tb7p.v6-dd.min.binpack.zst', 'test80-2022-09-sep-16tb7p.v6-dd.min.binpack.zst']VAL_Z = "test80-2022-08-aug-16tb7p.v6-dd.min.binpack.zst"DATA = "/kaggle/temp/data"os.makedirs(DATA, exist_ok=True)def fetch(name):    out = os.path.join(DATA, name[:-4])    if os.path.exists(out):        print("have", out)        return out    z = os.path.join(DATA, name)    # Fully quiet, deliberately. A progress bar redraws with carriage returns;    # on a tty that is one line updating in place, but a notebook cell is not a    # tty, so every redraw becomes a new line and gets captured into the    # notebook document. Three multi-GB downloads that way produce a log too    # large to open in a browser and an .ipynb to match.    # --rm deletes the archive as soon as it decompresses, which is what keeps    # three months inside Kaggle's ~70 GB working disk.    assert os.system(f"wget -q -O {z} {HOST}/{name}") == 0, name    assert os.system(f"zstd -d -q --rm {z} -o {out}") == 0, name    print(f"  {os.path.getsize(out) / 1e9:6.2f} GB  {out}", flush=True)    return outTRAIN_FILES = [fetch(n) for n in TRAIN_Z]VAL_FILE = fetch(VAL_Z)print(subprocess.run(["df", "-h", "/kaggle/temp"], capture_output=True,                     text=True).stdout)

## 5. Verify the data before training on a single position of itSame script, same thresholds as the local run. The number that matters is**score/result sign agreement**: if the evaluation and the outcome are bothside-to-move relative they agree ~95% of the time in decisive positions with aclear advantage, and if one were White-relative it would sit near 50% withnothing else in the file looking wrong. That is the check that caught aninverted convention in the Lichess extraction.Measured baselines. April (`.min`): pieces 18.71, result mean -0.0111, signagreement 0.946, unique 95.25%. June and September (`v6-dd.min`): pieces ~16.4,result mean ~-0.019, sign agreement **0.963**, unique **99.83%**.**If any file is REJECTED, stop - do not train on it.**

In [ ]:
import osos.environ["BTC_STREAM_EXE"] = "./binpack_stream"os.environ["BTC_VERIFY_ROWS"] = "2000000"!python verify_binpack.py {" ".join(TRAIN_FILES + [VAL_FILE])}

## 6. Train`BTC_STREAM_ROWS` sets how many positions count as one epoch. It also sets thelength of the cosine learning-rate schedule, which is why it has to be declaredrather than discovered - the stream does not know its own length until it hasfinished.

In [ ]:
import osos.environ["BTC_STREAM_EXE"] = "./binpack_stream"os.environ["BTC_KING_BUCKETS"] = "32"os.environ["BTC_OUT_BUCKETS"] = "8"os.environ["BTC_WDL_LAMBDA"] = "0.7"os.environ["BTC_STREAM_ROWS"] = "5000000000"os.environ["BTC_STREAM_VAL_ROWS"] = "4000000"os.environ["BTC_STREAM_BUFFER"] = "8000000"os.environ["BTC_VAL_BINPACK"] = VAL_FILEL1 = 512EPOCHS = 1# A directory argument makes the trainer stream every .binpack in it. One pass# over 5.6B positions is ~342,000 optimiser steps at batch 16384, which is# plenty for the cosine schedule to converge - fewer epochs over more unique# data beats more epochs over less, and more so the wider the net.!python nnue_train.py {DATA} /kaggle/working/net{L1} {L1} {EPOCHS}

## 7. Check the net before trusting itThe quantise step already asserts that nothing overflows int16. This re-readsthe saved file the way the engine will, so a net that cannot be loaded iscaught here rather than on the laptop.

In [ ]:
import numpy as npp = f"/kaggle/working/net{L1}/net.npz"d = np.load(p)for k in d.files:    print(f"  {k:14s} {str(d[k].dtype):8s} {d[k].shape}")acc = abs(d["ft_b"]).max() + 32 * abs(d["ft_w"]).max()print(f"worst-case accumulator {acc} of 32767  "      f"({'SAFE' if acc <= 32767 else 'OVERFLOW'})")print(f"file size {os.path.getsize(p) / 1e6:.2f} MB")print("50 MB cap:", "OK" if os.path.getsize(p) < 48e6 else "TOO BIG")

## 8. DownloadRight-click `net.npz` in the Output panel and save it. Put it in the repo rootand run `test_nnue.py`, `test_accumulator.py` and `test_mate.py` against itbefore it goes anywhere near a match.

In [ ]:
!ls -la /kaggle/working/net{L1}/